# demo_followup

> Follow-up runs after Phase 10 (speedup plan, `specs/multiagent/speedup/`): the round protocol and soft-territory squads (numpy, then LLM), a hard task board in the 30-70% solve band, and model size at matched effort; all through the faster scheduler.

Artifacts under `results/multiagent/followup/{quick,numpy,pilot,full}/` (gitignored): `runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md` and figures. The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), so one cap ($150) covers all live spend.

- **Quick** (normal tests): every follow-up path with the fake client; outputs identical for 1 and 8 processes, pooled and grid by grid.
- **B-numpy** (`--flags slow`): rounds and soft-territory squads with numpy agents at N 8-64; prints the bias chosen by the frozen rule.
- **D1 pilot** and **main** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live, only after `FOLLOWUP_PLAN` is frozen.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

from explRL.bench_grid import readCSV, runGrid, writeCSV
from explRL.followup_grid import (FOLLOWUP_PLAN, chooseBeta, chooseHard, effortGrid, followupHash, followupQuick, hardGrid,
                                  hardPilotGrid, roundsGrid, roundsNumpyGrid, runFollowupCell)
from explRL.followup_report import summaries, writeFigures, writeReport
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import quickGrid
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.team_time import teamRateCell, teamRateGrids
from explRL.train import demo_artifact_dir

DN = 'multiagent/followup'
WORKERS = min(8, os.cpu_count() or 1)          # fake-client and numpy workloads
LLM_WORKERS = 32                               # live workloads (decision SQ1; raised after the D1 pilot if retries allow)
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
P10 = LLM / 'full' / 'runs.csv'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def run(mode, grids, spec, p10, workers=None):
    "Run a workload through runWorkload; write runs.csv, turns.jsonl, timing.jsonl, report.md, figures, metrics.json."
    out = demo_artifact_dir(mode, DN)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    workers = workers or (LLM_WORKERS if spec.get('kind') == 'live' else WORKERS)
    t0 = time.time()
    res = runWorkload(grids, spec, workers, str(logs), run=runFollowupCell)
    rows, wall = res['rows'], time.time() - t0
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    llm = [r for r in rows if r['runner'] == 'llm']
    models = sorted({r['model'] for r in llm})
    meta = dict(mode=mode, planHash=followupHash(), commit=gitCommit(), models=models, cap=CAP, wall=wall,
                pricesChecked=', '.join(sorted({PRICES[m]['checked'] for m in models})) or '-',
                cost=float(sum(r['cost'] for r in llm)), calls=int(sum(r['calls'] for r in llm)), retries=timing['retries'],
                spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    arms = {}
    for key, sec in timing['runWall'].items():
        arms.setdefault(key.split('|')[2], []).append(sec)
    meta['runWall'] = {lv: sum(v) / len(v) for lv, v in arms.items()}
    S = summaries(rows, p10)
    writeReport(out / 'report.md', S, meta)
    writeFigures(out, rows, p10, S)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      pooled=res['pooled'], workers=workers, timing=timing,
                                                      plan=FOLLOWUP_PLAN), indent=2, sort_keys=True, default=str) + '\n')
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent "
          f"in all, {wall:.0f} s, {timing['callsPerSecond']:.1f} calls/s, {timing['retries']} retries; stopped: {res['stopped']}")
    return rows, S, meta, timing

## Quick: every follow-up path with the fake client (pipeline check)

In [ ]:
p10q = runWorkload([('q', [c for c in quickGrid() if c['study'] in ('main', 'sonnet')])], dict(kind='fake'), WORKERS)['rows']
t0 = time.time()
rows, S, meta, timing = run('quick', [('quick', followupQuick())], dict(kind='fake'), p10q)
assert time.time() - t0 < 30
q = demo_artifact_dir('quick', DN)
for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', 'hard.png', 'effort.png', 'rounds.png',
          'team_total.png'):
    assert (q / f).stat().st_size > 0, f
ref = (q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()
for w, pooled in ((1, True), (8, False)):
    run('quick', [('quick', followupQuick())], dict(kind='fake', pooled=pooled), p10q, workers=w)
    assert ((q / 'runs.csv').read_bytes(), (q / 'turns.jsonl').read_bytes()) == ref
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:1200])

## B-numpy: rounds and soft-territory squads (`--flags slow`)

Text-foraging world (5 s stays), blackboard, N 8-64, 600 s of world time, seeds 700-709; arms: event-driven, rounds of 5 s, rounds with squads of 2 at home bias 0, 0.25, 0.5, 1, 2 and infinity. The frozen rule picks the bias for B-LLM.

In [ ]:
#| slow
nrows, nS, _, _ = run('numpy', [('roundsNumpy', roundsNumpyGrid())], dict(kind='fake'), [])
print('home bias for B-LLM (frozen rule):', chooseBeta(nrows))

## Team rates for Phases 8-9 (`--flags slow`; FR-C3)

Reruns the Phase 8 main grid and the Phase 9 attack grid with `teamRateCell` into `results/multiagent/{bench,safety}/teamrate/` (never the confirmatory folders) and checks that every original column equals the saved CSV.

In [ ]:
#| slow
grids = teamRateGrids()
MA = demo_artifact_dir(None, 'multiagent')
for name, ref, study in (('bench', MA / 'bench/full/runs.csv', 'main'), ('safety', MA / 'safety/full/runs.csv', 'attack')):
    out = demo_artifact_dir('teamrate', f'multiagent/{name}') / 'runs.csv'
    writeCSV(runGrid(grids[name], WORKERS, run=teamRateCell), out)
    key = lambda r: tuple(r[k] for k in ('study', 'factor', 'level', 'channel', 'budget', 'N', 'seed'))
    old = {key(r): r for r in __import__('csv').DictReader(open(ref)) if r['study'] == study}
    new = {key(r): r for r in __import__('csv').DictReader(open(out))}
    assert set(old) == set(new) and all(o[c] == new[k][c] for k, o in old.items() for c in o), name
    print(name, len(new), 'rows: every original column identical')

## D1 pilot (`--flags llm`, live, about 360 calls): hard-board level and rate-limit probe

The board alone at difficulty 10 and 12 with `maxTokens` 8000 and 16 000 (seeds 640-642, 30 turns), at `LLM_WORKERS` processes; the timing summary is the rate-limit probe. The frozen rule picks the level; if none qualifies the notebook stops and the user decides.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FOLLOWUP_PLAN['frozen'], 'freeze and commit FOLLOWUP_PLAN before any live call'
    prows, _, _, ptiming = run('pilot', [('hardPilot', hardPilotGrid())], LIVE_SPEC, readCSV(P10))
    print('timing:', json.dumps(ptiming, indent=1, default=str)[:800])
    print('hard-board level (frozen rule):', chooseHard(prows))

## Main (`--cell-timeout 0 --flags llm`, live): D1, D2 and B-LLM in one workload

Priority order D1, D2, B-LLM (used only if the workload does not fit the cap). D1 runs only if the pilot found a level: on 2026-10-12 none qualified and the user paused D1 for a puzzle redesign.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FOLLOWUP_PLAN['frozen'], 'freeze and commit FOLLOWUP_PLAN before any live call'
    level = chooseHard(readCSV(demo_artifact_dir('pilot', DN) / 'runs.csv'))
    beta = chooseBeta(readCSV(demo_artifact_dir('numpy', DN) / 'runs.csv'))
    assert beta is not None, 'run B-numpy (--flags slow) first'
    # FR-D2: no level qualified in the pilot (solve rates 0.91-0.98), and the user chose (2026-10-12) to pause D1 and
    # redesign the puzzles in a later spec revision; D1 main runs only if a later pilot yields a level.
    grids = ([('hard', hardGrid(*level))] if level else []) + [('effort', effortGrid()), ('rounds', roundsGrid(beta))]
    print('D1 main:', level or 'paused (no qualifying level; user decision)', '| B-LLM home bias:', beta)
    rows, S, meta, timing = run('full', grids, LIVE_SPEC, readCSV(P10))